Get 3 closest topics in similarity to each topic 

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

with open("../../datasets/.pkl/window_topics.pkl", "rb") as f:
    window_topics = pickle.load(f)

In [5]:
def get_top_k_similar(source_emb, target_dict, k=3):
    if len(target_dict) == 0:
        return []

    target_ids = list(target_dict.keys())
    target_embs = np.vstack([target_dict[tid] for tid in target_ids])

    sims = cosine_similarity(source_emb.reshape(1, -1), target_embs)[0]
    top_indices = np.argsort(sims)[::-1][:k]

    return [target_ids[i] for i in top_indices]

In [6]:
rows = []
num_windows = len(window_topics)

for w_idx, window in enumerate(window_topics):
    current_topics = {
        tid: emb for tid, emb in window["topic_embeddings"].items()
        if tid != -1
    }

    for topic_id, topic_emb in current_topics.items():

        topic_label = f"{w_idx}_{topic_id}"

        # =============================
        # Successors 
        # =============================
        successors = []
        if w_idx < num_windows - 1:

            next_topics = {
                tid: emb for tid, emb in
                window_topics[w_idx + 1]["topic_embeddings"].items()
                if tid != -1
            }

            top_next = get_top_k_similar(topic_emb, next_topics, k=3)
            successors = [f"{w_idx+1}_{tid}" for tid in top_next]

        # =============================
        # Antecedents 
        # =============================
        antecedents = []
        if w_idx > 0:

            prev_topics = {
                tid: emb for tid, emb in
                window_topics[w_idx - 1]["topic_embeddings"].items()
                if tid != -1
            }

            top_prev = get_top_k_similar(topic_emb, prev_topics, k=3)
            antecedents = [f"{w_idx-1}_{tid}" for tid in top_prev]

        rows.append({
            "topic": topic_label,
            "successors": successors,
            "antecedents": antecedents
        })

transitions_df = pd.DataFrame(rows)

In [ ]:
transitions_df.to_csv("../../datasets/similartopics_auxiliardataset.csv", index=False)

In [ ]:
import pandas as pd

tweets1 = pd.read_csv("../../datasets/119congresstweets/bertilda.csv", encoding="utf-8", low_memory=False)
original = pd.read_csv("../../datasets/119congresstweets/bertilda_mergeandsplits.csv", encoding="utf-8", low_memory=False)
fowardonly = pd.read_csv("../../datasets/119congresstweets/bertilda_mergeandsplits_Forward-only.csv", encoding="utf-8", low_memory=False)
lexicalonly = pd.read_csv("../../datasets/GoldSet/bertilda_mergeandsplits_lexicalonly.csv", encoding="utf-8", low_memory=False)
nocoverage = pd.read_csv("../../datasets/GoldSet/bertilda_mergeandsplits_nocoverage.csv", encoding="utf-8", low_memory=False)
transitions_df = pd.read_csv("../../datasets/GoldSet/similartopics_auxiliardataset.csv", encoding="utf-8", low_memory=False)

In [36]:
import ast
def format_relation(row):

    rel = row["relation"]

    if rel == "continued":

        successors = row["successors"]

        if isinstance(successors, str):
            successors = ast.literal_eval(successors)

        return f"continued_{int(successors[0])}"

    if rel == "split":

        successors = row["successors"]

        if isinstance(successors, str):
            successors = ast.literal_eval(successors)

        successors = [str(int(t)) for t in successors]

        return f"split_{'_'.join(successors)}"


    if rel in ["disappeared", "unclear"]:
        return rel


    if rel == "merge":
        return "merge"

    return rel

In [37]:
import ast

tweets1["topics"] = tweets1["topics"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

tweets1["representative_tweets"] = tweets1["representative_tweets"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

transitions_df["successors"] = transitions_df["successors"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

transitions_df["antecedents"] = transitions_df["antecedents"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

relations_to_keep = ["split", "merge", "continued", "disappeared"]

original["relation_detailed"] = original.apply(format_relation, axis=1)

gold_base = (
    original[original["relation"].isin(relations_to_keep)]
    .groupby("relation")
    .sample(n=30, replace=False, random_state=42)
    .reset_index(drop=True)
)

In [ ]:
import ast

def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return x
    return x


def get_topic_info(window_id, topic_id):
    row = tweets1[tweets1["window_id"] == window_id]
    if row.empty:
        return None, None

    topics_list = safe_literal_eval(row.iloc[0]["topics"])
    tweets_list = safe_literal_eval(row.iloc[0]["representative_tweets"])

    if topic_id >= len(topics_list):
        return None, None

    top_words = topics_list[int(topic_id)]

    topic_tweets = tweets_list[int(topic_id)]

    if isinstance(topic_tweets, str):
        topic_tweets_split = [t.strip() for t in topic_tweets.split("||") if t.strip()]
    elif isinstance(topic_tweets, list):
        topic_tweets_split = []
        for t in topic_tweets:
            if isinstance(t, list):
                topic_tweets_split.extend(t)
            else:
                topic_tweets_split.append(t)
    else:
        topic_tweets_split = [str(topic_tweets)]


    rep_tweets = topic_tweets_split[:3]

    return top_words, rep_tweets


In [ ]:
gold_rows = []

for _, row in gold_base.iterrows():

    relation = row["relation"]

    if relation == "merge":
        w_case = int(row["window2"])
        t_case = int(row["topic_to"])
    else:
        w_case = int(row["window1"])
        t_case = int(row["topic1"])

    case_id = f"{w_case}_{t_case}"

    topic_words, topic_tweets = get_topic_info(w_case, t_case)

    trans_row = transitions_df[transitions_df["topic"] == case_id]

    formatted_adjacent_words = ""
    formatted_adjacent_tweets = ""

    if not trans_row.empty:

        if relation == "merge":
            adjacent_topics = trans_row.iloc[0]["antecedents"]
        else:
            adjacent_topics = trans_row.iloc[0]["successors"]

        for adj in adjacent_topics[:3]:

            w_adj, t_adj = map(int, adj.split("_"))

            words, tweets = get_topic_info(w_adj, t_adj)

            if words:
                if isinstance(words, list):
                    words_str = ", ".join(words)
                else:
                    words_str = str(words).replace("|", ", ")

                formatted_adjacent_words += (
                    f"Topic {t_adj}: {words_str}\n\n"
                )

            if tweets:
                formatted_adjacent_tweets += (
                    f"Topic {t_adj}:\n"
                )

                for i, tw in enumerate(tweets, 1):
                    formatted_adjacent_tweets += (
                        f"Tweet {i}: {tw}\n"
                    )

                formatted_adjacent_tweets += "\n"

    if isinstance(topic_words, list):
        topic_words_str = ", ".join(topic_words)
    else:
        topic_words_str = str(topic_words).replace("|", ", ")

    topic_tweets_str = ""
    if topic_tweets:
        for i, tw in enumerate(topic_tweets, 1):
            topic_tweets_str += f"Tweet {i}: {tw}\n"

    
    gold_rows.append({
        "case_study": case_id,
        "topic_top_words": topic_words_str,
        "topic_representative_tweets": topic_tweets_str,
        "adjacent_topics_top_words": formatted_adjacent_words,
        "adjacent_topics_representative_tweets": formatted_adjacent_tweets,
        "original": row["relation_detailed"]
    })

gold_df = pd.DataFrame(gold_rows)

In [40]:
gold_df

,case_study,topic_top_words,topic_representative_tweets,adjacent_topics_top_words,adjacent_topics_representative_tweets,original
0,106_0,"donald, trump, community, work, make, family, ...",Tweet 1: Thank you to Governor DeSantis for de...,"Topic 0: donald, trump, community, work, famil...",Topic 0:\nTweet 1: Stop 2 of my Made in Americ...,continued_0
1,26_9,"navy, army, game, football, championship, beat...",Tweet 1: Love to see that people are flocking ...,"Topic 14: championship, football, sport, state...","Topic 14:\nTweet 1: As a proud Padre, I couldn...",continued_14
2,117_8,"hispanic, month, heritage, contribution, cultu...",Tweet 1: #HispanicHeritageMonth is a time to c...,"Topic 8: hispanic, month, heritage, contributi...",Topic 8:\nTweet 1: #Miami is the Capital of La...,continued_8
3,83_19,"ban, travel, trump, donald, bigotry, security,...","Tweet 1: Once again, President Trump has issue...","Topic 18: ban, travel, trump, donald, security...",Topic 18:\nTweet 1: TSA is ridiculous. Just sa...,continued_18
4,48_1,"vladimir, ukraine, putin, russia, war, donald,...",Tweet 1: I’m LIVE on the Senate floor urging c...,"Topic 1: vladimir, ukraine, russia, putin, war...",Topic 1:\nTweet 1: Today marks three years sin...,continued_1
...,...,...,...,...,...,...
115,16_5,"health, care, medicare, healthcare, coverage, ...",Tweet 1: I was pleased to meet Dr. Saitta of F...,"Topic 22: medicaid, medicare, health, care, cu...",Topic 22:\nTweet 1: .@HouseDemocrats fought to...,split_10_22
116,62_2,"great, student, community, meet, farmer, farm,...",Tweet 1: Hardworking producers are the backbon...,"Topic 2: farmer, great, farm, discuss, support...",Topic 2:\nTweet 1: It was an honor to accept t...,split_2_4
117,117_7,"energy, environmental, climate, protection, cl...","Tweet 1: Today, the @EnergyCommerce Energy sub...","Topic 22: energy, grid, permit, clean, power, ...",Topic 22:\nTweet 1: Establishing a more afford...,split_19_22
118,50_3,"border, illegal, alien, fentanyl, immigration,...","Tweet 1: 🚨PROMISES MADE, PROMISES KEPT!\n\nCom...","Topic 11: border, illegal, biden, joe, crossin...","Topic 11:\nTweet 1: Under President Biden, cri...",split_11_22


Annotator Excel

In [6]:
import pandas as pd

annotation_df = gold_df.copy()

if "relation_type" in annotation_df.columns:
    annotation_df = annotation_df.drop(columns=["relation_type"])      

annotation_df["Antecedents/Sucessors"] = ""  
annotation_df["Main"] = ""   
annotation_df["Confidence"] = ""  

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation

df = gold_df.copy()

if "relation_type" in df.columns:
    df = df.drop(columns=["relation_type"])

df["Antecedents/Sucessors"] = ""
df["Antecedents/Sucessors 2"] = ""  
df["Main"] = ""
df["Confidence"] = ""

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

wb = Workbook()
ws = wb.active
ws.title = "Annotation"

ws.append(list(df.columns))

for r in df.itertuples(index=False):
    ws.append(list(r))


dv_conf = DataValidation(type="list", formula1='"1,2,3"', allow_blank=True)
ws.add_data_validation(dv_conf)
for row in range(2, len(df)+2):
    dv_conf.add(ws[f"H{row}"])  

# -----------------------------
# Validation: Antecedents/Sucessors Main 
# -----------------------------
for row in range(2, len(df)+2):
    topics_str = ws[f"D{row}"].value  
    topics = [line.split(":")[0].strip() for line in topics_str.split("\n") if line.strip()]
    options = ["None"] + topics
    options_formula = '"' + ",".join(options) + '"'
    dv = DataValidation(type="list", formula1=options_formula, allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(ws[f"F{row}"])  

# -----------------------------
# Validation: Main 
# -----------------------------
for row in range(2, len(df)+2):
    topics_str = ws[f"D{row}"].value
    topics = [line.split(":")[0].strip() for line in topics_str.split("\n") if line.strip()]
    options = ["None"] + topics
    options_formula = '"' + ",".join(options) + '"'
    dv = DataValidation(type="list", formula1=options_formula, allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(ws[f"G{row}"])  

# -----------------------------
# Validation: Antecedents/Sucessors 2 
# -----------------------------
for row in range(2, len(df)+2):
    topics_str = ws[f"D{row}"].value
    topics = [line.split(":")[0].strip() for line in topics_str.split("\n") if line.strip()]
    options = ["None"] + topics
    options_formula = '"' + ",".join(options) + '"'
    dv = DataValidation(type="list", formula1=options_formula, allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(ws[f"I{row}"])

wb.save("goldset.xlsx")

Creation of the dataset with the variations

In [41]:
import ast

def get_relation_variation(case_row, variation_df):

    original_relation = case_row["original"]
    case_id = case_row["case_study"]

    w_case, t_case = map(int, case_id.split("_"))

    if original_relation != "merge":

        match = variation_df[
            (variation_df["window1"] == w_case) &
            (variation_df["topic1"] == t_case)
        ]

        if not match.empty:
            return format_relation(match.iloc[0])

        else:
            return "merge"

    else:

        orig_row = original[
            (original["window2"] == w_case) &
            (original["topic_to"] == t_case)
        ]

        if orig_row.empty:
            return "error"

        row_data = orig_row.iloc[0]
        window_from = int(row_data["window1"])

        topics_from = row_data["topics_from"]

        if isinstance(topics_from, str):
            topics_from = ast.literal_eval(topics_from)

        results = []

        for t_from in topics_from:

            match = variation_df[
                (variation_df["window1"] == window_from) &
                (variation_df["topic1"] == int(t_from))
            ]

            if not match.empty:
                formatted = format_relation(match.iloc[0])
                results.append(f"{window_from}_{t_from}_{formatted}")
            else:
                results.append(f"{window_from}_{t_from}_merge")

        return " | ".join(results)

In [42]:
gold_df["forward_only"] = gold_df.apply(
    lambda row: get_relation_variation(row, fowardonly),
    axis=1
)

gold_df["lexical_only"] = gold_df.apply(
    lambda row: get_relation_variation(row, lexicalonly),
    axis=1
)

gold_df["no_coverage"] = gold_df.apply(
    lambda row: get_relation_variation(row, nocoverage),
    axis=1
)

In [43]:
gold_df

,case_study,topic_top_words,topic_representative_tweets,adjacent_topics_top_words,adjacent_topics_representative_tweets,original,forward_only,lexical_only,no_coverage
0,106_0,"donald, trump, community, work, make, family, ...",Tweet 1: Thank you to Governor DeSantis for de...,"Topic 0: donald, trump, community, work, famil...",Topic 0:\nTweet 1: Stop 2 of my Made in Americ...,continued_0,continued_0,disappeared,continued_0
1,26_9,"navy, army, game, football, championship, beat...",Tweet 1: Love to see that people are flocking ...,"Topic 14: championship, football, sport, state...","Topic 14:\nTweet 1: As a proud Padre, I couldn...",continued_14,continued_14,disappeared,continued_14
2,117_8,"hispanic, month, heritage, contribution, cultu...",Tweet 1: #HispanicHeritageMonth is a time to c...,"Topic 8: hispanic, month, heritage, contributi...",Topic 8:\nTweet 1: #Miami is the Capital of La...,continued_8,continued_8,continued_8,continued_8
3,83_19,"ban, travel, trump, donald, bigotry, security,...","Tweet 1: Once again, President Trump has issue...","Topic 18: ban, travel, trump, donald, security...",Topic 18:\nTweet 1: TSA is ridiculous. Just sa...,continued_18,continued_18,unclear,unclear
4,48_1,"vladimir, ukraine, putin, russia, war, donald,...",Tweet 1: I’m LIVE on the Senate floor urging c...,"Topic 1: vladimir, ukraine, russia, putin, war...",Topic 1:\nTweet 1: Today marks three years sin...,continued_1,continued_1,disappeared,continued_1
...,...,...,...,...,...,...,...,...,...
115,16_5,"health, care, medicare, healthcare, coverage, ...",Tweet 1: I was pleased to meet Dr. Saitta of F...,"Topic 22: medicaid, medicare, health, care, cu...",Topic 22:\nTweet 1: .@HouseDemocrats fought to...,split_10_22,split_10_22,disappeared,split_10_22
116,62_2,"great, student, community, meet, farmer, farm,...",Tweet 1: Hardworking producers are the backbon...,"Topic 2: farmer, great, farm, discuss, support...",Topic 2:\nTweet 1: It was an honor to accept t...,split_2_4,split_2_4,disappeared,unclear
117,117_7,"energy, environmental, climate, protection, cl...","Tweet 1: Today, the @EnergyCommerce Energy sub...","Topic 22: energy, grid, permit, clean, power, ...",Topic 22:\nTweet 1: Establishing a more afford...,split_19_22,split_19_22,disappeared,split_19_22
118,50_3,"border, illegal, alien, fentanyl, immigration,...","Tweet 1: 🚨PROMISES MADE, PROMISES KEPT!\n\nCom...","Topic 11: border, illegal, biden, joe, crossin...","Topic 11:\nTweet 1: Under President Biden, cri...",split_11_22,split_11_22,disappeared,split_11_22


In [44]:
def simplify_merge_result(row_value):

    if pd.isna(row_value):
        return row_value

    parts = row_value.split(" | ")

    relations = []

    for p in parts:
        tokens = p.split("_")

        if "continued" in tokens:
            idx = tokens.index("continued")

            topic_id = tokens[idx - 1]

            return f"continued_{topic_id}"

        relations.append(tokens[-1])

    if "merge" in relations and "disappeared" in relations:
        return "disappeared"

    if all(r in ["disappeared", "unclear"] for r in relations):
        return "disappeared"

    if all(r == "merge" for r in relations):
        return "merge"

    return row_value

In [45]:
variation_columns = ["forward_only", "lexical_only", "no_coverage"]

for col in variation_columns:

    mask = gold_df["original"] == "merge"

    gold_df.loc[mask, col] = (
        gold_df.loc[mask, col]
        .apply(simplify_merge_result)
    )

In [46]:
gold_df

,case_study,topic_top_words,topic_representative_tweets,adjacent_topics_top_words,adjacent_topics_representative_tweets,original,forward_only,lexical_only,no_coverage
0,106_0,"donald, trump, community, work, make, family, ...",Tweet 1: Thank you to Governor DeSantis for de...,"Topic 0: donald, trump, community, work, famil...",Topic 0:\nTweet 1: Stop 2 of my Made in Americ...,continued_0,continued_0,disappeared,continued_0
1,26_9,"navy, army, game, football, championship, beat...",Tweet 1: Love to see that people are flocking ...,"Topic 14: championship, football, sport, state...","Topic 14:\nTweet 1: As a proud Padre, I couldn...",continued_14,continued_14,disappeared,continued_14
2,117_8,"hispanic, month, heritage, contribution, cultu...",Tweet 1: #HispanicHeritageMonth is a time to c...,"Topic 8: hispanic, month, heritage, contributi...",Topic 8:\nTweet 1: #Miami is the Capital of La...,continued_8,continued_8,continued_8,continued_8
3,83_19,"ban, travel, trump, donald, bigotry, security,...","Tweet 1: Once again, President Trump has issue...","Topic 18: ban, travel, trump, donald, security...",Topic 18:\nTweet 1: TSA is ridiculous. Just sa...,continued_18,continued_18,unclear,unclear
4,48_1,"vladimir, ukraine, putin, russia, war, donald,...",Tweet 1: I’m LIVE on the Senate floor urging c...,"Topic 1: vladimir, ukraine, russia, putin, war...",Topic 1:\nTweet 1: Today marks three years sin...,continued_1,continued_1,disappeared,continued_1
...,...,...,...,...,...,...,...,...,...
115,16_5,"health, care, medicare, healthcare, coverage, ...",Tweet 1: I was pleased to meet Dr. Saitta of F...,"Topic 22: medicaid, medicare, health, care, cu...",Topic 22:\nTweet 1: .@HouseDemocrats fought to...,split_10_22,split_10_22,disappeared,split_10_22
116,62_2,"great, student, community, meet, farmer, farm,...",Tweet 1: Hardworking producers are the backbon...,"Topic 2: farmer, great, farm, discuss, support...",Topic 2:\nTweet 1: It was an honor to accept t...,split_2_4,split_2_4,disappeared,unclear
117,117_7,"energy, environmental, climate, protection, cl...","Tweet 1: Today, the @EnergyCommerce Energy sub...","Topic 22: energy, grid, permit, clean, power, ...",Topic 22:\nTweet 1: Establishing a more afford...,split_19_22,split_19_22,disappeared,split_19_22
118,50_3,"border, illegal, alien, fentanyl, immigration,...","Tweet 1: 🚨PROMISES MADE, PROMISES KEPT!\n\nCom...","Topic 11: border, illegal, biden, joe, crossin...","Topic 11:\nTweet 1: Under President Biden, cri...",split_11_22,split_11_22,disappeared,split_11_22


In [ ]:
gold_df.to_csv("../../datasets/GoldSet/goldvariationsrelations.csv", index=False)

Evaluation

In [ ]:
import pandas as pd
variations = pd.read_csv("../../datasets/GoldSet/goldvariationsrelations.csv", encoding="utf-8", low_memory=False)
annotator1 = pd.read_excel("../../datasets/GoldSet/GoldSet - annotator1.xlsx")
annotator2 = pd.read_excel("../../datasets/GoldSet/GoldSet - annotator2.xlsx")
annotator3 = pd.read_excel("../../datasets/GoldSet/GoldSet - annotator3.xlsx")

In [ ]:
import pandas as pd
import numpy as np
import re

def extract_topic_number(value):
    """Extrai número de 'Topic X'."""
    if pd.isna(value):
        return None

    match = re.search(r"Topic\s*(\d+)", str(value))
    return match.group(1) if match else None


def transform_annotator(df, variations, annotator_name):
    
    variation_lookup = (
        variations.set_index("case_study")["original"]
        .to_dict()
    )

    def classify(row):

        t1 = extract_topic_number(row["Antecedents/Sucessors"])
        t2 = extract_topic_number(row["Antecedents/Sucessors 2"])

        case = row["case_study"]

        # --- disappeared ---
        if t1 is None and t2 is None:
            return "disappeared"

        # --- continued ---
        if t1 is not None and t2 is None:
            return f"continued_{t1}"

        # --- merge / split ---
        if t1 is not None and t2 is not None:

            original_type = variation_lookup.get(case)

            if original_type == "merge":
                return "merge"
            else:
                return f"split_{t1}_{t2}"

        return np.nan

    df[annotator_name] = df.apply(classify, axis=1)

    return df

In [5]:
annotator1 = transform_annotator(annotator1, variations, "annotator1")
annotator2 = transform_annotator(annotator2, variations, "annotator2")
annotator3 = transform_annotator(annotator3, variations, "annotator3")

In [6]:
var_cols = [
    "case_study",
    "original",
    "forward_only",
    "lexical_only",
    "no_coverage",
    "adjacent_topics_top_words"
]

variations_small = variations[var_cols].copy()

variations_small["merge_key"] = (
    variations_small["case_study"].astype(str) + "_" +
    variations_small["adjacent_topics_top_words"].astype(str)
)
annotator1["merge_key"] = (
    annotator1["case_study"].astype(str) + "_" +
    annotator1["adjacent_topics_top_words"].astype(str)
)
annotator2["merge_key"] = (
    annotator2["case_study"].astype(str) + "_" +
    annotator2["adjacent_topics_top_words"].astype(str)
)
annotator3["merge_key"] = (
    annotator3["case_study"].astype(str) + "_" +
    annotator3["adjacent_topics_top_words"].astype(str)
)

def prepare_annotator(df, annotator_name, confidence_name):
    return (
        df[["merge_key", annotator_name, "Confidence"]]
        .rename(columns={"Confidence": confidence_name})
    )

ann1 = prepare_annotator(annotator1, "annotator1", "Confidence_1")
ann2 = prepare_annotator(annotator2, "annotator2", "Confidence_2")
ann3 = prepare_annotator(annotator3, "annotator3", "Confidence_3")

final_dataset = (
    variations_small
    .merge(ann1, on="merge_key", how="left")
    .merge(ann2, on="merge_key", how="left")
    .merge(ann3, on="merge_key", how="left")
)

final_dataset = final_dataset.drop(columns=["merge_key", "adjacent_topics_top_words"])

In [7]:
final_dataset

,case_study,original,forward_only,lexical_only,no_coverage,annotator1,Confidence_1,annotator2,Confidence_2,annotator3,Confidence_3
0,106_0,continued_0,continued_0,disappeared,continued_0,continued_0,2,split_0_1,2,continued_0,3
1,26_9,continued_14,continued_14,disappeared,continued_14,continued_14,3,split_14_15,1,continued_14,3
2,117_8,continued_8,continued_8,continued_8,continued_8,continued_8,3,continued_8,3,continued_8,3
3,83_19,continued_18,continued_18,unclear,unclear,continued_18,3,continued_18,3,split_0_18,3
4,48_1,continued_1,continued_1,disappeared,continued_1,continued_1,3,continued_1,3,continued_1,3
...,...,...,...,...,...,...,...,...,...,...,...
115,16_5,split_10_22,split_10_22,disappeared,split_10_22,split_22_10,1,split_22_10,3,split_22_10,2
116,62_2,split_2_4,split_2_4,disappeared,unclear,split_2_4,1,split_2_4,2,disappeared,1
117,117_7,split_19_22,split_19_22,disappeared,split_19_22,split_22_19,2,split_22_19,2,split_22_19,2
118,50_3,split_11_22,split_11_22,disappeared,split_11_22,split_11_22,2,split_11_22,2,split_11_22,2


In [8]:
def event_type(value):
    if pd.isna(value):
        return None
    
    value = str(value)

    if value.startswith("continued"):
        return "continued"
    if value.startswith("split"):
        return "split"
    if value == "merge":
        return "merge"
    if value == "disappeared":
        return "disappeared"
    
    return None

df = final_dataset.copy()

df["original_type"] = df["original"].apply(event_type)
df["ann1_type"] = df["annotator1"].apply(event_type)
df["ann2_type"] = df["annotator2"].apply(event_type)
df["ann3_type"] = df["annotator3"].apply(event_type)

df["all_correct"] = (
    (df["ann1_type"] == df["original_type"]) &
    (df["ann2_type"] == df["original_type"]) &
    (df["ann3_type"] == df["original_type"])
)

accuracy_per_event = (
    df.groupby("original_type")["all_correct"]
    .mean()
    .reset_index(name="accuracy")
)

df["n_correct"] = (
    (df["ann1_type"] == df["original_type"]).astype(int) +
    (df["ann2_type"] == df["original_type"]).astype(int) +
    (df["ann3_type"] == df["original_type"]).astype(int)
)

df["majority_correct"] = df["n_correct"] >= 2

majority_accuracy = (
    df.groupby("original_type")["majority_correct"]
    .mean()
    .reset_index(name="accuracy_majority")
)

summary = (
    df.groupby("original_type")
    .agg(
        total_cases=("case_study", "count"),
        unanimous_accuracy=("all_correct", "mean"),
        majority_accuracy=("majority_correct", "mean")
    )
)

print(summary)

               total_cases  unanimous_accuracy  majority_accuracy
original_type                                                    
continued               30            0.533333           0.866667
disappeared             30            0.566667           0.800000
merge                   30            0.400000           0.666667
split                   30            0.366667           0.766667


In [9]:
def event_type(value):
    if pd.isna(value):
        return None
    
    value = str(value)

    if value.startswith("continued"):
        return "continued"
    if value.startswith("split"):
        return "split"
    if value == "merge":
        return "merge"
    if value == "disappeared":
        return "disappeared"
    
    return None

df2 = final_dataset.copy()

df2["forward_only_type"] = df2["forward_only"].apply(event_type)
df2["ann1_type"] = df2["annotator1"].apply(event_type)
df2["ann2_type"] = df2["annotator2"].apply(event_type)
df2["ann3_type"] = df2["annotator3"].apply(event_type)

df2["all_correct"] = (
    (df2["ann1_type"] == df2["forward_only_type"]) &
    (df2["ann2_type"] == df2["forward_only_type"]) &
    (df2["ann3_type"] == df2["forward_only_type"])
)

accuracy_per_event = (
    df2.groupby("forward_only_type")["all_correct"]
    .mean()
    .reset_index(name="accuracy")
)

df2["n_correct"] = (
    (df2["ann1_type"] == df2["forward_only_type"]).astype(int) +
    (df2["ann2_type"] == df2["forward_only_type"]).astype(int) +
    (df2["ann3_type"] == df2["forward_only_type"]).astype(int)
)

df2["majority_correct"] = df2["n_correct"] >= 2

majority_accuracy = (
    df2.groupby("forward_only_type")["majority_correct"]
    .mean()
    .reset_index(name="accuracy_majority")
)

summary = (
    df2.groupby("forward_only_type")
    .agg(
        total_cases=("case_study", "count"),
        unanimous_accuracy=("all_correct", "mean"),
        majority_accuracy=("majority_correct", "mean")
    )
)

print(summary)

                   total_cases  unanimous_accuracy  majority_accuracy
forward_only_type                                                    
continued                   35            0.457143           0.800000
disappeared                 43            0.395349           0.558140
merge                       14            0.428571           0.714286
split                       23            0.391304           0.826087


In [ ]:
import pandas as pd

def event_type(value):
    """Converte valores como 'continued_3' ou 'split_2_4' em categorias puras."""
    if pd.isna(value):
        return None
    value = str(value)
    if value.startswith("continued"):
        return "continued"
    if value.startswith("split"):
        return "split"
    if value == "merge":
        return "merge"
    if value == "disappeared":
        return "disappeared"
    return None

def evaluate_against_annotators(df, reference_col, annotator_cols=["annotator1", "annotator2", "annotator3"]):
    """
    Calcula resumo de acerto unânime e por maioria comparando os annotators com uma coluna de referência.
    
    df: DataFrame
    reference_col: coluna que serve de "truth" (ex: 'forward_only', 'lexical_only', 'no_coverage')
    annotator_cols: lista das colunas dos annotators
    """
    df = df.copy()
    
    df["reference_type"] = df[reference_col].apply(event_type)
    for col in annotator_cols:
        df[col + "_type"] = df[col].apply(event_type)
    
    df["n_correct"] = df[[col + "_type" for col in annotator_cols]].eq(df["reference_type"], axis=0).sum(axis=1)
    
    df["unanimous_correct"] = df["n_correct"] == len(annotator_cols)
    
    majority_threshold = len(annotator_cols) // 2 + 1
    df["majority_correct"] = df["n_correct"] >= majority_threshold
    
    summary = (
        df.groupby("reference_type")
        .agg(
            total_cases=("reference_type", "count"),
            unanimous_accuracy=("unanimous_correct", "mean"),
            majority_accuracy=("majority_correct", "mean")
        )
    )
    
    return summary

In [ ]:
summary_forward = evaluate_against_annotators(final_dataset, "original")
print("BERTilda summary:\n", summary_forward.round(3))

print("\n ----------------------------------------------------------- \n")


summary_forward = evaluate_against_annotators(final_dataset, "forward_only")
print("Forward-only summary:\n", summary_forward.round(3))

print("\n ----------------------------------------------------------- \n")

summary_lexical = evaluate_against_annotators(final_dataset, "lexical_only")
print("Lexical-only summary:\n", summary_lexical.round(3))

print("\n ----------------------------------------------------------- \n")

summary_nocov = evaluate_against_annotators(final_dataset, "no_coverage")
print("No-coverage summary:\n", summary_nocov.round(3))

BERTilda summary:
                 total_cases  unanimous_accuracy  majority_accuracy
reference_type                                                    
continued                30               0.533              0.867
disappeared              30               0.567              0.800
merge                    30               0.400              0.667
split                    30               0.367              0.767

 ----------------------------------------------------------- 

Forward-only summary:
                 total_cases  unanimous_accuracy  majority_accuracy
reference_type                                                    
continued                35               0.457              0.800
disappeared              43               0.395              0.558
merge                    14               0.429              0.714
split                    23               0.391              0.826

 ----------------------------------------------------------- 

Lexical-only summary:
    

In [12]:
reference_columns = {
    "BERTilda": "original",
    "Similarity-only": "no_coverage",
    "Lexical-only": "lexical_only",
    "Forward-only": "forward_only"
}

summary_dict = {}
for name, col in reference_columns.items():
    summary_dict[name] = evaluate_against_annotators(final_dataset, col)

summary_table = pd.DataFrame(index=["continued", "disappeared", "split", "merge"])
for name, summary in summary_dict.items():
    summary_table[name] = summary["unanimous_accuracy"]

summary_table = summary_table.round(3)

print("Unanimous accuracy table (all 3 annotators agree) by event:\n")
print(summary_table)

Unanimous accuracy table (all 3 annotators agree) by event:

             BERTilda  Similarity-only  Lexical-only  Forward-only
continued       0.533            0.550          0.75         0.457
disappeared     0.567            0.312          0.16         0.395
split           0.367            0.381           NaN         0.391
merge           0.400            0.300           NaN         0.429


In [13]:
import pandas as pd

# Reference columns: BERTilda (original) and the baseline/ablation methods
reference_columns = {
    "BERTilda": "original",
    "Similarity-only": "no_coverage",
    "Lexical-only": "lexical_only",
    "Forward-only": "forward_only"
}

# Evaluate each reference column against the 3 annotators
summary_dict = {}
for name, col in reference_columns.items():
    summary_dict[name] = evaluate_against_annotators(final_dataset, col)

# Create a summary table with event types as rows and methods as columns
majority_table = pd.DataFrame(index=["continued", "disappeared", "split", "merge"])
for name, summary in summary_dict.items():
    majority_table[name] = summary["majority_accuracy"]

# Round values to 3 decimal places
majority_table = majority_table.round(3)

print("Majority accuracy table (at least 2 out of 3 annotators agree) by event:\n")
print(majority_table)

Majority accuracy table (at least 2 out of 3 annotators agree) by event:

             BERTilda  Similarity-only  Lexical-only  Forward-only
continued       0.867            0.750         0.750         0.800
disappeared     0.800            0.417         0.226         0.558
split           0.767            0.667           NaN         0.826
merge           0.667            0.450           NaN         0.714


In [20]:
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

annotations = df[["annotator1", "annotator2", "annotator3"]]

categories = sorted(pd.unique(annotations.values.ravel()))

matrix = []

for _, row in annotations.iterrows():
    counts = [(row == cat).sum() for cat in categories]
    matrix.append(counts)

matrix = np.array(matrix)

kappa = fleiss_kappa(matrix)

print("Fleiss' kappa:", kappa)

Fleiss' kappa: 0.6111464648156212


In [26]:
import krippendorff
import pandas as pd
import numpy as np

annot_cols = ["annotator1", "annotator2", "annotator3"]

all_labels = pd.unique(df[annot_cols].values.ravel())

label_map = {label: i for i, label in enumerate(all_labels)}

coded = df[annot_cols].apply(lambda col: col.map(label_map))

reliability_data = coded.T.values

alpha = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement='nominal'  
)

print("Krippendorff's alpha:", alpha)

Krippendorff's alpha: 0.6122266135244666
